# 🇨🇳 RAGScore — 中文QA生成 & RAG评估演示

RAGScore 自动检测中文文档，并使用中文生成QA对和评估RAG系统。

| 功能 | 说明 |
|------|------|
| 🌍 自动语言检测 | 检测中文字符，使用中文提示词 |
| 🎯 `--audience` | 根据目标受众定制问题 |
| 📋 `--purpose` | 根据文档用途定制问题 |
| 🔍 `--diagnose` | 失败根因分析（检索缺失/幻觉/不完整/误解） |

**需要：** `ragscore >= 0.8.3`、OpenAI API Key（或其他支持的提供商）

## 1. 安装

In [ ]:
!pip install -q "ragscore[notebook]>=0.8.3" openai numpy

import nest_asyncio
nest_asyncio.apply()
print("✅ 准备完成")

## 2. API Key 设置

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ 从 Colab Secrets 读取了密钥")
except Exception:
    os.environ["OPENAI_API_KEY"] = "sk-..."  # 在此输入你的密钥
    print("⚠️  使用硬编码密钥 — 推荐使用 Colab Secrets")

## 3. 创建中文示例文档

In [ ]:
%%writefile chinese_doc.txt
大型语言模型（LLM）是基于深度学习的自然语言处理模型，通过在海量文本数据上进行预训练来学习语言模式和知识。代表性的模型包括OpenAI的GPT系列、Google的Gemini、Meta的LLaMA以及国内的通义千问、文心一言等。这些模型的参数规模从数十亿到数千亿不等，展现了涌现能力——即在模型规模达到一定阈值后突然出现的新能力。

检索增强生成（RAG）是一种将信息检索与文本生成相结合的技术架构。传统的LLM存在知识截止日期和幻觉问题，即模型可能生成看似合理但实际错误的信息。RAG通过在生成前检索相关文档，将外部知识注入到生成过程中，有效缓解了这些问题。RAG系统通常包含三个核心组件：文档索引（将文档切分并向量化存储）、检索器（根据查询找到相关文档片段）和生成器（基于检索到的上下文生成回答）。

向量数据库是RAG系统的关键基础设施。主流的向量数据库包括Pinecone、Weaviate、Milvus、Qdrant和Chroma。这些数据库使用近似最近邻（ANN）算法实现高效的向量相似度搜索。常用的ANN算法包括HNSW（Hierarchical Navigable Small World）和IVF（Inverted File Index）。向量数据库的选择取决于数据规模、查询延迟要求和部署环境。对于百万级文档，Milvus和Pinecone表现优异；对于原型开发，Chroma因其轻量级特性而广受欢迎。

RAG系统的评估是一个重要但复杂的问题。评估维度包括：检索质量（检索到的文档是否包含正确答案的支持证据）、生成质量（生成的回答是否正确、完整且忠实于检索到的内容）、端到端性能（最终回答是否满足用户需求）。RAGScore等工具通过自动生成QA数据集并使用LLM作为评判者来实现自动化评估，大大降低了人工评估的成本。评估指标通常包括准确率、完整性、相关性、简洁性和忠实度。

分块策略对RAG系统性能有显著影响。常见的分块方法包括：固定长度分块（按token数或字符数切分）、语义分块（基于句子边界或段落切分）、递归分块（先按大结构切分，再逐级细化）。理想的分块大小通常在200-500个token之间，过大会引入噪声，过小会丢失上下文。重叠窗口（通常10-20%）可以减少信息在边界处的丢失。此外，元数据增强（为每个分块添加标题、来源等信息）和多级索引（摘要级+细节级）也是提升检索质量的有效手段。

在企业级RAG部署中，安全性和隐私保护至关重要。敏感数据不应发送到外部API，因此本地部署的LLM（如通过Ollama运行的LLaMA或Qwen模型）成为重要选择。访问控制需要在文档级别实施——不同用户只能检索到其有权限的文档。此外，RAG系统需要处理多语言场景、实时更新的知识库、以及与企业现有系统（如Confluence、SharePoint、Notion）的集成。生产环境中还需考虑可观测性，包括查询日志、检索召回率监控、回答质量趋势等。

## 4. 语言自动检测测试

In [ ]:
from ragscore.llm import detect_language

with open("chinese_doc.txt") as f:
    text = f.read()

lang = detect_language(text)
print(f"检测到的语言: {lang}")  # 应显示 'zh'
assert lang == "zh", f"期望: 'zh', 实际: '{lang}'"
print("✅ 中文被正确检测")

## 5. 构建迷你 RAG

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI()

with open("chinese_doc.txt") as f:
    text = f.read()
chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 50]
print(f"✅ 创建了 {len(chunks)} 个分块")

def get_embedding(text):
    return client.embeddings.create(input=text, model="text-embedding-3-small").data[0].embedding

chunk_embeddings = np.array([get_embedding(c) for c in chunks])
print(f"✅ 已嵌入 {len(chunk_embeddings)} 个分块")

def my_rag(question):
    """嵌入问题 → 检索前3个分块 → GPT-4o 生成回答"""
    q_emb = np.array(get_embedding(question))
    sims = chunk_embeddings @ q_emb / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb)
    )
    top_idx = np.argsort(sims)[-3:][::-1]
    context = "\n\n".join([chunks[i] for i in top_idx])

    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "请仅基于提供的上下文回答问题。简洁且准确。"},
            {"role": "user", "content": f"上下文:\n{context}\n\n问题: {question}"},
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content

print("\n🧪 功能验证:")
print(my_rag("什么是检索增强生成？"))

## 6. 中文 RAG 评估（基线）

In [ ]:
from ragscore import quick_test

result = quick_test(my_rag, docs="chinese_doc.txt", n=5)
print("\n📋 生成的问题:")
for _, row in result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")

## 7. 🎯 目标受众：AI 工程师

In [ ]:
engineer_result = quick_test(
    my_rag, docs="chinese_doc.txt", n=5,
    audience="AI工程师和MLOps从业者",
    purpose="技术选型和架构设计",
)
print("\n🔧 AI工程师向问题:")
for _, row in engineer_result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")

## 8. 🎯 目标受众：产品经理

In [ ]:
pm_result = quick_test(
    my_rag, docs="chinese_doc.txt", n=5,
    audience="产品经理和业务决策者",
    purpose="产品规划和商业决策",
)
print("\n💼 产品经理向问题:")
for _, row in pm_result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")

## 9. 🔍 失败诊断 (`--diagnose`)

使用 `--diagnose` 了解每个错误回答的根本原因：

In [ ]:
# 诊断功能通过 CLI 使用：
# ragscore evaluate http://localhost:8000/query --diagnose
#
# 输出示例：
# 🔍 Failure Diagnosis:
#   Retriever Miss: 3 (42.9%)
#   Generator Hallucination: 2 (28.6%)
#   Incomplete Answer: 1 (14.3%)
#   Wrong Interpretation: 1 (14.3%)
#
# 也可与 --detailed 组合使用：
# ragscore evaluate http://localhost:8000/query --diagnose --detailed -o results.json

print("🔍 --diagnose 功能说明：")
print("  • Retriever Miss（检索缺失）: RAG未能检索到包含答案证据的文档块")
print("  • Generator Hallucination（生成器幻觉）: 检索正确但生成了虚假信息")
print("  • Incomplete Answer（不完整回答）: 检索正确但回答不完整")
print("  • Wrong Interpretation（错误解读）: 检索正确但曲解了内容")
print("\n💡 提示: 使用 'ragscore generate docs/ -o golden.jsonl' 生成的 QA 已包含 support_span")
print("   然后运行 'ragscore evaluate <endpoint> -g golden.jsonl --diagnose' 获取诊断报告")

## 10. 结果对比

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "目标受众": ["基线", "AI工程师", "产品经理"],
    "准确率": [
        f"{result.accuracy:.0%}",
        f"{engineer_result.accuracy:.0%}",
        f"{pm_result.accuracy:.0%}",
    ],
    "平均分数": [
        f"{result.avg_score:.1f}/5",
        f"{engineer_result.avg_score:.1f}/5",
        f"{pm_result.avg_score:.1f}/5",
    ],
    "示例问题": [
        result.df.iloc[0]["question"][:60] + "..." if len(result.df) > 0 else "N/A",
        engineer_result.df.iloc[0]["question"][:60] + "..." if len(engineer_result.df) > 0 else "N/A",
        pm_result.df.iloc[0]["question"][:60] + "..." if len(pm_result.df) > 0 else "N/A",
    ],
})
display(comparison)

---

## 📚 资源

- **GitHub**: https://github.com/HZYAI/RagScore
- **PyPI**: https://pypi.org/project/ragscore/
- **网站**: https://ragscore.io

⭐ 如果 RAGScore 对你有帮助，请在 GitHub 上给我们一颗星！